# K-Means Clustering Model Training

Train a K-Means clustering model for email spam detection (unsupervised).

## Notebook Structure
1. Import modules
2. Load dataset
3. Split data
4. Create and train pipeline
5. Evaluate clustering
6. Visualize results
7. Save model (.pkl)

In [3]:
import sys
import os
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from src.config import DATA_PATHS, MODEL_OUTPUT_DIR
from src.load_data import load_dataset
from src.utils import split_data
from src.train_kmeans import (
    create_pipeline,
    train,
    save_model,
    predict,
    get_cluster_centers,
    evaluate_clustering,
    print_clustering_results
)

In [4]:
# Load Dataset
df = load_dataset(dataset_name="250K_email_dataset")

print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nFirst few samples:")
print(df.head())

Dataset shape: (322601, 2)

Label distribution:
label
0    168454
1    154147
Name: count, dtype: int64

First few samples:
                                                text  label
0  wrong bill grace i ' ll forward original messa...      0
1  i have continued the hilcorp old ocean deal da...      0
2  several related issues have resulted in an inc...      0
3  one year rate for this one will be escapenumbe...      0
4  attached is the weekly deal report for escapen...      0


In [ ]:
# Split Data
X_train, X_test, y_train, y_test = split_data(
    texts=df['text'].values,
    labels=df['label'].values,
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining set label distribution:")
print(f"  Ham: {sum(y_train == 0)}")
print(f"  Spam: {sum(y_train == 1)}")
print(f"\nTest set label distribution:")
print(f"  Ham: {sum(y_test == 0)}")
print(f"  Spam: {sum(y_test == 1)}")

In [ ]:
# Create and Train Model
pipeline = create_pipeline(
    n_clusters=3,
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words='english',
    random_state=42
)

print("Training K-Means pipeline...")
pipeline = train(pipeline, X_train)
print("Training completed!")

In [ ]:
# Save Model
model_path = MODEL_OUTPUT_DIR / "kmeans_model.pkl"
save_model(pipeline, model_path)
print(f"Model saved to: {model_path}")

In [ ]:
# Evaluate Clustering
y_pred = predict(pipeline, X_test)

# Calculate metrics
metrics = evaluate_clustering(y_test, y_pred)
print_clustering_results(metrics)

# Get cluster centers (top words)
print("\nTop words per cluster:")
cluster_centers = get_cluster_centers(pipeline, n_terms=10)
for cluster_id, words in cluster_centers.items():
    print(f"\nCluster {cluster_id}:")
    print(f"  {', '.join(words)}")